# 🔬 Low-SNR Image Processing: CryoEM Tutorial

A hands-on walkthrough of signal processing techniques for **extremely low SNR** images,
using cryoEM as the primary application.

---

### Topics
1. SNR intuition & noise sources
2. CryoEM forward model (phantom + CTF + noise)
3. Classical denoising (Gaussian, Median, Wiener, Bandpass)
4. CTF estimation & correction
5. Particle picking via NCC template matching
6. 2D class averaging
7. Fourier Ring Correlation (FRC) & resolution
8. Noise2Void self-supervised denoising concept
9. Full benchmark

### Install
```bash
pip install numpy scipy matplotlib scikit-image tqdm
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import ndimage, signal, fft
from scipy.ndimage import gaussian_filter, median_filter
from scipy.signal import wiener
from skimage import filters, feature, transform
from skimage.draw import disk
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'image.cmap':       'gray',
    'font.size':        11,
})

print('All imports OK')

---
## 1. Why Is CryoEM So Hard? — SNR Intuition

In cryoEM, the electron dose must stay **extremely low** to avoid radiation damage.
Typical micrograph SNR: **0.01–0.1** — signal buried 10–100x below noise.

Noise sources:
- **Shot noise (Poisson)** — discrete electron counting
- **Read-out noise (Gaussian)** — detector electronics
- **Structural noise** — amorphous ice background
- **CTF modulation** — defocus smears & inverts frequency bands

Key insight: **average** N identical copies → SNR improves by √N.

In [ ]:
def snr_db(signal_img, noisy_img):
    noise = noisy_img - signal_img
    return 10 * np.log10(np.var(signal_img) / (np.var(noise) + 1e-12))

size = 128
x = np.linspace(-3, 3, size)
xx, yy = np.meshgrid(x, x)
clean_signal = np.exp(-(xx**2 + yy**2) / 2)

noise_levels = [0.1, 0.5, 1.0, 3.0, 10.0]
fig, axes = plt.subplots(1, len(noise_levels) + 1, figsize=(14, 3))
axes[0].imshow(clean_signal)
axes[0].set_title('Clean signal')
axes[0].axis('off')
for ax, sigma in zip(axes[1:], noise_levels):
    noisy = clean_signal + np.random.normal(0, sigma, clean_signal.shape)
    db = snr_db(clean_signal, noisy)
    ax.imshow(noisy, vmin=-3*sigma, vmax=3*sigma)
    ax.set_title(f'sigma={sigma}\nSNR={db:.1f} dB')
    ax.axis('off')
plt.suptitle('Effect of Increasing Noise', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Simulating CryoEM-Like Images

Forward model pipeline:
1. Particle phantom (ring-shaped protein complex)
2. CTF modulation in Fourier space
3. Additive Gaussian + shot noise

### Contrast Transfer Function (CTF)

$$\text{CTF}(s) = -\sin\!\left(\pi \lambda \Delta z \, s^2 - \frac{\pi}{2}\lambda^3 C_s \, s^4\right)$$

where $\lambda$ = electron wavelength, $\Delta z$ = defocus, $C_s$ = spherical aberration.

In [ ]:
def make_particle_phantom(size=128, n_subunits=6, subunit_radius=8):
    img = np.zeros((size, size))
    cx, cy = size // 2, size // 2
    ring_r = size // 5      # radius of the subunit ring (≈25 px)
    for k in range(n_subunits):
        angle = 2 * np.pi * k / n_subunits      # evenly space 6 subunits around the ring
        sx = int(cx + ring_r * np.cos(angle))
        sy = int(cy + ring_r * np.sin(angle))
        rr, cc = disk((sy, sx), subunit_radius, shape=img.shape)
        img[rr, cc] += np.exp(-((rr-sy)**2 + (cc-sx)**2) / (2*(subunit_radius/2)**2))
    rr, cc = disk((cy, cx), subunit_radius // 2, shape=img.shape)
    img[rr, cc] += 0.5
    return img / img.max()


def compute_ctf(size, defocus_um=1.5, voltage_kv=300, cs_mm=2.7,
                amplitude_contrast=0.1, pixel_size_A=1.5):
    lam = 12.26 / np.sqrt(voltage_kv*1e3 * (1 + voltage_kv*1e3 / (2*511e3)))  # relativistic de Broglie wavelength (Å)
    defocus = defocus_um * 1e4
    cs = cs_mm * 1e7
    freq = np.fft.fftfreq(size, d=pixel_size_A)
    fx, fy = np.meshgrid(freq, freq)
    s2 = fx**2 + fy**2
    chi = np.pi * lam * defocus * s2 - 0.5 * np.pi * (lam**3) * cs * s2**2   # phase error as function of spatial freq s
    return -np.sqrt(1 - amplitude_contrast**2) * np.sin(chi) - amplitude_contrast * np.cos(chi) # q = amplitude contrast (0.1)
#The -sin(chi) dominates (phase contrast). The -q·cos(chi) is a small amplitude contrast correction.

def apply_ctf_and_noise(phantom, ctf_arr, snr=0.1):
    F = fft.fft2(phantom)
    img_ctf = np.real(fft.ifft2(F * fft.ifftshift(ctf_arr)))  # multiply by CTF, back to real space
    noise_sigma = np.sqrt(np.var(img_ctf) / snr)   # scale noise to achieve target SNR
    return img_ctf, img_ctf + np.random.normal(0, noise_sigma, img_ctf.shape)


N = 128
phantom = make_particle_phantom(N)
ctf     = compute_ctf(N, defocus_um=1.5)
img_ctf, noisy_img = apply_ctf_and_noise(phantom, ctf, snr=0.05)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, t in zip(axes,
    [phantom, fft.fftshift(ctf), img_ctf, noisy_img],
    ['Phantom', 'CTF (Fourier)', 'After CTF', '+ Noise (SNR=0.05)']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('CryoEM Forward Model', fontsize=13)
plt.tight_layout(); plt.show()
print(f'Measured SNR: {snr_db(img_ctf, noisy_img):.2f} dB')

---
## 3. Classical Denoising Methods

| Method | Domain | Notes |
|---|---|---|
| Gaussian | Spatial | Smooths all frequencies, blurs edges |
| Median | Spatial | Non-linear, robust to impulsive noise |
| Wiener | Frequency | Optimal MMSE under Gaussian noise |
| Bandpass | Fourier | Standard cryoEM preprocessing |

In [ ]:
def bandpass_filter(img, low_freq=0.02, high_freq=0.4):
    F = fft.fftshift(fft.fft2(img))
    h, w = img.shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x-cx)**2 + (y-cy)**2) / (min(h,w) / 2)
    order = 4
    lp = 1 / (1 + (r / high_freq)**(2*order))
    hp = 1 - 1 / (1 + (r / low_freq)**(2*order))
    return np.real(fft.ifft2(fft.ifftshift(F * lp * hp)))


g1 = gaussian_filter(noisy_img, sigma=1)
g2 = gaussian_filter(noisy_img, sigma=2)
med3 = median_filter(noisy_img, size=3)
med5 = median_filter(noisy_img, size=5)
wien = wiener(noisy_img, mysize=7)
bp   = bandpass_filter(noisy_img, low_freq=0.02, high_freq=0.35)

results = {
    'Noisy Input': noisy_img, 'Gaussian s=1': g1, 'Gaussian s=2': g2,
    'Median 3x3': med3, 'Median 5x5': med5, 'Wiener 7x7': wien,
    'Bandpass': bp, 'Ground Truth': img_ctf,
}

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (title, img) in zip(axes.flat, results.items()):
    ax.imshow(img, cmap='gray')
    if title not in ('Noisy Input', 'Ground Truth'):
        ax.set_title(f'{title}\n{snr_db(img_ctf,img):.1f} dB')
    else:
        ax.set_title(title)
    ax.axis('off')
plt.suptitle('Classical Denoising Comparison', fontsize=13)
plt.tight_layout(); plt.show()

print('SNR summary (vs noisy baseline {:.1f} dB):'.format(snr_db(img_ctf, noisy_img)))
for k, v in list(results.items())[1:-1]:
    print(f'  {k:15s}: {snr_db(img_ctf,v):+.2f} dB')

---
## 4. CTF Estimation & Correction

The CTF introduces **phase reversals** at specific spatial frequencies.
Without correction, averaging reinforces destructive interference.

- **Phase-flipping**: flip sign of Fourier components where CTF < 0
- **Wiener CTF correction**: optimal linear filter

$$\hat{F}(s) = \frac{\text{CTF}(s)}{\text{CTF}^2(s) + \varepsilon} \cdot G(s)$$

In [ ]:
def phase_flip_ctf(img, ctf_arr):
    F = fft.fft2(img)
    c = fft.ifftshift(ctf_arr)
    sgn = np.sign(c); sgn[sgn == 0] = 1
    return np.real(fft.ifft2(F * sgn))


def wiener_ctf(img, ctf_arr, eps=0.1):
    F = fft.fft2(img)
    c = fft.ifftshift(ctf_arr)
    return np.real(fft.ifft2((c / (c**2 + eps)) * F))


pf = phase_flip_ctf(noisy_img, ctf)
wc = wiener_ctf(noisy_img, ctf, eps=0.2)

ctf_1d  = ctf[N//2, N//2:]
freq_1d = np.fft.fftfreq(N)[N//2:]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].plot(freq_1d, ctf_1d, color='#4fc3f7', lw=2)
axes[0].axhline(0, color='white', lw=0.5, ls='--')
axes[0].set_xlabel('Spatial freq (1/px)'); axes[0].set_ylabel('CTF')
axes[0].set_title('1-D CTF Profile\nDz=1.5um, 300kV')
axes[0].set_facecolor('#16213e')

for ax, img, title in zip(axes[1:], [noisy_img, pf, wc],
                           ['Noisy+CTF', 'Phase-flipped', 'Wiener CTF']):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.suptitle('CTF Correction Methods', fontsize=13)
plt.tight_layout(); plt.show()

---
## 5. Simulated Micrograph & Particle Picking

Simulate a 512x512 micrograph with 30 randomly placed, randomly rotated particles
in an ice background, then locate them with **Normalized Cross-Correlation (NCC)**.

In [ ]:
def make_micrograph(size=512, n_particles=30, particle_size=32, snr=0.05):
    mic = np.zeros((size, size))
    template_clean = make_particle_phantom(particle_size)
    ice_bg = gaussian_filter(np.random.randn(size, size), sigma=15) * 0.3
    mic += ice_bg
    centres = []
    half = particle_size // 2
    for _ in range(n_particles):
        angle = np.random.uniform(0, 360)
        rotated = transform.rotate(template_clean, angle, resize=False)
        y = np.random.randint(half, size - half)
        x = np.random.randint(half, size - half)
        mic[y-half:y+half, x-half:x+half] += rotated
        centres.append((y, x))
    ctf_mic = compute_ctf(size, defocus_um=np.random.uniform(0.8, 2.5))
    _, mic_noisy = apply_ctf_and_noise(mic, ctf_mic, snr=snr)
    return mic_noisy, centres, template_clean


def ncc_pick(micrograph, template, threshold=0.2):
    from skimage.feature import match_template, peak_local_max
    result = match_template(micrograph, template, pad_input=True)
    peaks = peak_local_max(result, min_distance=template.shape[0]//2,
                           threshold_abs=threshold)
    return result, peaks


mic, true_centres, templ = make_micrograph()
cc_map, detected = ncc_pick(mic, templ)

radius = 16
true_arr = np.array(true_centres)
tp = sum(any(np.sqrt((d[0]-t[0])**2+(d[1]-t[1])**2)<radius for t in true_arr)
         for d in detected)
prec = tp / max(len(detected), 1)
rec  = tp / max(len(true_centres), 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(mic, cmap='gray')
for y,x in true_centres:
    axes[0].add_patch(plt.Circle((x,y), radius, color='lime', fill=False, lw=1.2))
axes[0].set_title(f'Ground Truth ({len(true_centres)} particles)'); axes[0].axis('off')

axes[1].imshow(cc_map, cmap='hot')
axes[1].set_title('NCC Cross-Correlation Map'); axes[1].axis('off')

axes[2].imshow(mic, cmap='gray')
for y,x in detected:
    axes[2].add_patch(plt.Circle((x,y), radius, color='cyan', fill=False, lw=1.2))
axes[2].set_title(f'Detected Picks\nPrec={prec:.2f}  Recall={rec:.2f}'); axes[2].axis('off')

plt.suptitle('Particle Picking via NCC Template Matching', fontsize=13)
plt.tight_layout(); plt.show()

---
## 6. 2D Class Averaging

The fundamental SNR-boosting technique: align N particles to a reference, then average.

$$\text{SNR}_{\text{avg}} = N \times \text{SNR}_{\text{single}}$$

Algorithm: extract boxes → rotational alignment → average → repeat.

In [ ]:
def extract_particles(micrograph, centres, box_size=32):
    half = box_size // 2
    H, W = micrograph.shape
    out = []
    for y, x in centres:
        if half <= y < H-half and half <= x < W-half:
            box = micrograph[y-half:y+half, x-half:x+half]
            box = (box - box.mean()) / (box.std() + 1e-8)
            out.append(box)
    return np.array(out)

"""
The key trick is warp_polar — it remaps the 2D image into polar coordinates (r, θ), so that a rotation in Cartesian space becomes a horizontal shift in polar space. Finding the best rotation then reduces to a standard 1D cross-correlation problem, which is fast and reliable.
"""
def rot_align(particle, reference):
    from skimage.transform import warp_polar
    r = min(particle.shape) // 2
    pp = warp_polar(particle,  radius=r)
    rp = warp_polar(reference, radius=r)
    cc = signal.fftconvolve(pp, rp[::-1,::-1], mode='full')
    best = np.unravel_index(np.argmax(cc), cc.shape)[0]
    angle = (best - pp.shape[0]) * (360 / pp.shape[0])
    return transform.rotate(particle, -angle, resize=False)
"""
a bootstrap loop
Each iteration does two things:

Align every particle to the current reference
Build a new reference by averaging all aligned particles
"""
def class_average(particles, n_iter=4):
    ref = particles.mean(axis=0) # iteration 0: unaligned average
    history = [ref.copy()]
    for _ in range(n_iter):
        aligned = np.array([rot_align(p, ref) for p in particles])
        ref = aligned.mean(axis=0)  # new reference from aligned particles
        history.append(ref.copy())
    return ref, history


particles = extract_particles(mic, true_centres, box_size=32)
print(f'Extracted {len(particles)} particles')
avg, history = class_average(particles, n_iter=4)

ref_rs = transform.resize(templ, (32, 32))
ref_norm = (ref_rs - ref_rs.mean()) / (ref_rs.std() + 1e-8)

n_show = min(5, len(particles))
fig, axes = plt.subplots(2, n_show+1, figsize=(14, 6))
for i in range(n_show):
    axes[0,i].imshow(particles[i], cmap='gray')
    axes[0,i].set_title(f'Particle #{i+1}'); axes[0,i].axis('off')
axes[0,-1].imshow(ref_norm, cmap='gray')
axes[0,-1].set_title('Ground Truth'); axes[0,-1].axis('off')
for i, h in enumerate(history[:n_show]):
    h2 = h[:ref_norm.shape[0], :ref_norm.shape[1]]
    axes[1,i].imshow(h2, cmap='gray')
    axes[1,i].set_title(f'Iter {i}  {snr_db(ref_norm, h2):.1f} dB')
    axes[1,i].axis('off')
axes[1,-1].axis('off')
plt.suptitle('2D Class Averaging', fontsize=13)
plt.tight_layout(); plt.show()
h, w = ref_norm.shape
print(f'Final class avg SNR : {snr_db(ref_norm, avg[:h,:w]):.2f} dB')
print(f'Single particle SNR : {snr_db(ref_norm, particles[0][:h,:w]):.2f} dB')

---
## 7. Fourier Ring Correlation (FRC) & Resolution

**FRC** is the gold standard for 2D resolution estimation:

$$\text{FRC}(r) = \frac{\sum_{\mathbf{k}\in r} F_1(\mathbf{k}) F_2^*(\mathbf{k})}{\sqrt{\sum |F_1|^2 \cdot \sum |F_2|^2}}$$

Resolution is reported at FRC = **0.143** (half-bit criterion).

In [ ]:
def compute_frc(img1, img2):
    F1 = fft.fftshift(fft.fft2(img1))
    F2 = fft.fftshift(fft.fft2(img2))
    h, w = img1.shape
    cy, cx = h//2, w//2
    y, x = np.indices(img1.shape)
    r = np.sqrt((x-cx)**2 + (y-cy)**2).astype(int)
    r_max = min(cx, cy)
    frc_vals = np.zeros(r_max)
    for ri in range(r_max):
        mask = r == ri
        num   = np.real(np.sum(F1[mask] * np.conj(F2[mask])))
        denom = np.sqrt(np.sum(np.abs(F1[mask])**2)*np.sum(np.abs(F2[mask])**2)+1e-12)
        frc_vals[ri] = num / denom
    return frc_vals


half_n = len(particles) // 2
frc = compute_frc(particles[:half_n].mean(axis=0),
                  particles[half_n:].mean(axis=0))
freqs = np.arange(len(frc)) / (len(frc) * 2)
psd = np.abs(fft.fftshift(fft.fft2(mic)))**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.imshow(np.log1p(psd), cmap='inferno')
ax1.set_title('Log Power Spectral Density'); ax1.axis('off')

ax2.plot(freqs, frc, color='#4fc3f7', lw=2, label='FRC')
ax2.axhline(0.143, color='#ff6b6b', ls='--', lw=1.5, label='0.143 (half-bit)')
ax2.axhline(0.5,   color='#ffd93d', ls='--', lw=1.5, label='0.5')
ax2.fill_between(freqs, frc, 0, alpha=0.15, color='#4fc3f7')
ax2.set_xlabel('Spatial Frequency (cycles/pixel)')
ax2.set_ylabel('FRC'); ax2.set_ylim(-0.2, 1.1)
ax2.set_title('Fourier Ring Correlation Curve')
ax2.set_facecolor('#16213e')
below = np.where(frc < 0.143)[0]
if len(below):
    ax2.axvline(freqs[below[0]], color='#a29bfe', ls=':', lw=1.5,
                label=f'Cutoff: {freqs[below[0]]:.3f} cyc/px')
ax2.legend()
plt.suptitle('PSD and FRC Resolution Analysis', fontsize=13)
plt.tight_layout(); plt.show()

---
## 8. Self-Supervised Denoising — Noise2Void Concept

No clean cryoEM targets exist, so supervised denoising is impossible.
**Noise2Void** sidesteps this:

1. Mask ~2-5% of pixels per training patch
2. Predict each masked pixel from its **neighbours only** (blind-spot)
3. Since noise is i.i.d., neighbours carry zero info about the masked pixel's noise
4. Network is forced to learn the underlying signal structure

Scenario A: The mask falls on the Background (Pure Noise) expectation at 0
Scenario B: The mask falls on a True Protein (Signal) refer to neighboring pixel
Real tools: `noise2void` (pip), `cryoCARE`, `Topaz-Denoise`

In [ ]:
def n2v_demo(image, mask_frac=0.05, nbhd=3):
    H, W = image.shape
    masked   = image.copy().astype(float)
    replaced = image.copy().astype(float)
    n_mask = int(H * W * mask_frac)
    ys = np.random.randint(nbhd, H-nbhd, n_mask)
    xs = np.random.randint(nbhd, W-nbhd, n_mask)
    for y, x in zip(ys, xs):
        masked[y, x] = np.nan
        patch = image[y-nbhd:y+nbhd+1, x-nbhd:x+nbhd+1].copy().astype(float)
        patch[nbhd, nbhd] = np.nan
        replaced[y, x] = np.nanmean(patch)
    return masked, replaced


def shift_avg(image, n=16, shift=2):
    acc = np.zeros_like(image, dtype=float)
    for k in range(n):
        a = 2 * np.pi * k / n
        acc += np.roll(np.roll(image, int(round(shift*np.sin(a))), axis=0),
                       int(round(shift*np.cos(a))), axis=1)
    return acc / n


p = particles[0] if len(particles) > 0 else noisy_img[:32, :32]
masked_p, n2v_p = n2v_demo(p)
s_avg = shift_avg(p)

fig, axes = plt.subplots(1, 4, figsize=(13, 4))
for ax, img, title in zip(axes,
    [p, masked_p, n2v_p, s_avg],
    ['Noisy particle', 'N2V masked', 'N2V neighbour-fill', 'Shift avg (16x)']):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.suptitle('Noise2Void: Blind-Spot Masking Concept', fontsize=13)
plt.tight_layout(); plt.show()

print('N2V Key Idea')
print('--------------------------------------------')
print('- Mask ~2-5 percent of pixels per patch')
print('- Predict from neighbours ONLY (blind-spot)')
print('- i.i.d. noise: neighbours give no info about')
print('  masked center -> network learns signal')
print('- No clean reference needed!')

---
## 9. Comprehensive Benchmark

All methods compared on the same noisy particle with SNR in dB.

In [ ]:
tp_ = particles[0] if len(particles) > 0 else noisy_img[:32, :32]
h, w = tp_.shape
tc_ = ref_norm[:h, :w]

methods = {
    'Raw noisy':    tp_,
    'Gaussian s=1': gaussian_filter(tp_, sigma=1),
    'Gaussian s=2': gaussian_filter(tp_, sigma=2),
    'Median 3x3':   median_filter(tp_, size=3),  #Median is non-linear and good for spike noise, but in cryoEM the noise is Gaussian distributed
    'Median 5x5':   median_filter(tp_, size=5),
    'Wiener':       wiener(tp_, mysize=5),  #Wiener wins because it's mathematically optimal — it minimises mean squared error under Gaussian noise
    'Bandpass':     bandpass_filter(tp_),
    'Class avg':    avg[:h, :w],
}

snrs = {k: snr_db(tc_, v[:h,:w]) for k, v in methods.items()}

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios':[1,1.5]})

colors = plt.cm.cool(np.linspace(0.1, 0.9, len(snrs)))
bars = axes[0].barh(list(snrs.keys()), list(snrs.values()), color=colors)
axes[0].axvline(snrs['Raw noisy'], color='#ff6b6b', ls='--', lw=1.5, label='Baseline')
axes[0].set_xlabel('SNR (dB)'); axes[0].set_title('SNR — All Methods'); axes[0].legend()
for bar, v in zip(bars, snrs.values()):
    axes[0].text(v+0.05, bar.get_y()+bar.get_height()/2,
                 f'{v:.1f}', va='center', fontsize=9)

axes[1].axis('off')
plt.suptitle('Comprehensive Denoising Benchmark', fontsize=13)
plt.tight_layout(); plt.show()

fig2, axes2 = plt.subplots(2, 4, figsize=(13, 6))
for ax, (k, v) in zip(axes2.flat, methods.items()):
    ax.imshow(v[:h,:w], cmap='gray')
    ax.set_title(f'{k}\n{snrs[k]:.2f} dB', fontsize=9)
    ax.axis('off')
plt.suptitle('Visual Comparison', fontsize=13)
plt.tight_layout(); plt.show()

best = max(snrs, key=snrs.get)
print(f'Best method: {best} ({snrs[best]:.2f} dB)')

What looks better visuallyWhat is actually closer to ground truthSmooth, blurred (Gaussian s=2)Grainy but structure-preserving (Class avg)
This is exactly why cryoEM practitioners never judge reconstruction quality by eye alone — FRC curves exist precisely because visual smoothness is a misleading proxy for real resolution. A map that looks beautiful but fails FRC at 8 Å is worse than a grainier-looking one that passes at 4 Å.

---
## 10. Key Takeaways & Next Steps

| Concept | Key Insight |
|---|---|
| **Forward model** | CTF + Poisson + Gaussian noise |
| **CTF correction** | Phase-flip first; Wiener CTF for amplitude |
| **Classical filters** | Bandpass & Wiener best for mild denoising |
| **Class averaging** | SNR scales as √N — the gold standard |
| **FRC** | 0.143 half-bit criterion for resolution |
| **Self-supervised** | Noise2Void / cryoCARE — no clean targets needed |

---
$$
\text{average} = \frac{1}{N}\sum_{i=1}^{N}(\text{signal}+\text{noise}_i)
= \text{signal} + \frac{1}{N}\sum_{i}\text{noise}_i
$$

This is why modern cryoEM datasets collect millions of particles
---

### FRC — "0.143 half-bit criterion for resolution"

This answers the question: *how do you know when your class average is actually showing real structure vs just correlated noise?*

The procedure is to split your particles into two independent halves, compute a class average from each half separately, then measure how similar those two averages are frequency-by-frequency:

$$\text{FRC}(r) = \frac{\sum_{\mathbf{k} \in r} F_1(\mathbf{k}) \cdot F_2^*(\mathbf{k})}{\sqrt{\sum|F_1|^2 \cdot \sum|F_2|^2}}$$

This gives a correlation value between 0 and 1 at each spatial frequency ring. The logic is:

- If both halves agree at a given frequency (FRC ≈ 1) → that frequency contains **reproducible signal**
- If both halves disagree (FRC ≈ 0) → that frequency is **dominated by noise**, which is different in each half

The **0.143 threshold** comes from information theory — it's the point where the two half-maps together contain exactly 1 bit of information per Fourier coefficient (the "half-bit" criterion). Below this threshold you are essentially fitting noise. It's conservative by design — better to underreport resolution than to claim structure that isn't there.

The critical insight is that **FRC measures reproducibility, not visual quality**. A blurred, over-smoothed map might look beautiful but fail FRC at low resolution because the smoothing is hiding the fact that the two halves never agreed at high frequencies. This is why FRC is the standard and not a subjective sharpness score.
---
